In [5]:
# ============================================
# CELL 1 - IMPORT LIBRARIES
# ============================================

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# ============================================
# CELL 2 - MAIN CRACK DETECTION FUNCTION
# ============================================

def detect_crack(image_path):

    # ========================================
    # 1. Load Image
    # ========================================

    img = cv2.imread(image_path)

    if img is None:
        return None


    # ========================================
    # 2. Resize Image
    # ========================================

    resized = cv2.resize(
        img,
        (640, 640)
    )


    # ========================================
    # 3. Windshield ROI
    # ========================================

    roi = resized[
        100:460,
        25:615
    ]


    # ========================================
    # 4. Grayscale Conversion
    # ========================================

    gray = cv2.cvtColor(
        roi,
        cv2.COLOR_BGR2GRAY
    )


    # ========================================
    # 5. Bilateral Filtering
    # ========================================

    bilateral = cv2.bilateralFilter(
        gray,
        d=5,
        sigmaColor=35,
        sigmaSpace=40
    )


    # ========================================
    # 6. Adaptive Canny Edge Detection
    # ========================================

    median = np.median(
        bilateral
    )

    sigma = 0.33

    lower = int(
        max(
            0,
            (1.0 - sigma) * median
        )
    )

    upper = int(
        min(
            255,
            (1.0 + sigma) * median
        )
    )

    edges_canny = cv2.Canny(
        bilateral,
        lower,
        upper
    )


    # ========================================
    # 7. Morphological Closing
    # ========================================

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    canny_morph = cv2.morphologyEx(
        edges_canny,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=1
    )


    # ========================================
    # 8. Contour Detection
    # ========================================

    contours, _ = cv2.findContours(
        canny_morph,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )


    # ========================================
    # 9. Calculate Edge Density
    # ========================================

    edge_pixels = cv2.countNonZero(
        canny_morph
    )

    total_pixels = (
        canny_morph.shape[0]
        * canny_morph.shape[1]
    )

    edge_density = (
        edge_pixels / total_pixels
    )


    # ========================================
    # 10. Count Long Contours
    # ========================================

    long_contour_count = 0

    for contour in contours:

        length = cv2.arcLength(
            contour,
            True
        )

        if length > 100:
            long_contour_count += 1


    # ========================================
    # 11. Detect Valid Crack Regions
    # ========================================

    min_area = 30
    max_area = 1800
    min_length = 60
    margin = 10
    max_extent = 0.25

    valid_cracks = []

    for contour in contours:

        area = cv2.contourArea(
            contour
        )

        length = cv2.arcLength(
            contour,
            True
        )

        x, y, w, h = cv2.boundingRect(
            contour
        )

        if w == 0 or h == 0:
            continue

        extent = (
            area / (w * h)
        )

        if (
            area > min_area
            and area < max_area
            and length > min_length
            and extent < max_extent
            and x > margin
            and y > margin
            and (x + w) < (
                roi.shape[1] - margin
            )
            and (y + h) < (
                roi.shape[0] - margin
            )
        ):

            valid_cracks.append(
                contour
            )


    # ========================================
    # 12. Candidate Region Image
    # Small boxes before final big box
    # ========================================

    candidate_img = roi.copy()

    for contour in valid_cracks:

        x, y, w, h = cv2.boundingRect(
            contour
        )

        cv2.rectangle(
            candidate_img,
            (x, y),
            (x + w, y + h),
            (0, 0, 255),
            2
        )


    # ========================================
    # 13. Final Decision
    # ========================================

    edge_density_threshold = 0.08
    min_valid_regions = 1
    min_long_contours = 2

    if (
        edge_density >= edge_density_threshold
        and len(valid_cracks) >= min_valid_regions
        and long_contour_count >= min_long_contours
    ):

        prediction = 1
        status = "CRACK DETECTED"
        text_color = (0, 0, 255)

    else:

        prediction = 0
        status = "NO CRACK DETECTED"
        text_color = (0, 255, 0)


    # ========================================
    # 14. Final Result Image
    # One big box around detected regions
    # ========================================

    result_img = roi.copy()

    if (
        prediction == 1
        and len(valid_cracks) > 0
    ):

        all_points = np.vstack(
            valid_cracks
        )

        x, y, w, h = cv2.boundingRect(
            all_points
        )

        cv2.rectangle(
            result_img,
            (x, y),
            (x + w, y + h),
            (0, 0, 255),
            3
        )


    # ========================================
    # Add Final Status Text
    # ========================================

    cv2.putText(
        result_img,
        status,
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        text_color,
        3
    )


    # ========================================
    # Return All Results
    # ========================================

    return {
        "prediction": prediction,
        "status": status,

        "roi": roi,
        "gray": gray,
        "bilateral": bilateral,
        "canny": edges_canny,
        "morph": canny_morph,

        "candidate_img": candidate_img,
        "result_img": result_img,

        "edge_density": edge_density,
        "valid_regions": len(valid_cracks),
        "long_contours": long_contour_count,

        "lower_threshold": lower,
        "upper_threshold": upper
    }


print("detect_crack() function created successfully.")

detect_crack() function created successfully.


In [7]:
# ============================================
# CELL 3 - SINGLE IMAGE DEMONSTRATION
# ============================================

image_path = "Dataset_Finalised/Cracked/TestCracked1.jpg" ### Put Image Sample Here / LETAK GAMBAR DEKAT SINI

results = detect_crack(
    image_path
)

if results is None:
    raise FileNotFoundError(
        "Image not found."
    )


# ============================================
# DISPLAY ALL PROCESSING STAGES
# ============================================

plt.figure(
    figsize=(18, 10)
)


# 1. Windshield ROI
plt.subplot(2, 4, 1)

plt.imshow(
    cv2.cvtColor(
        results["roi"],
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    "1. Windshield ROI"
)

plt.axis("off")


# 2. Grayscale
plt.subplot(2, 4, 2)

plt.imshow(
    results["gray"],
    cmap="gray"
)

plt.title(
    "2. Grayscale"
)

plt.axis("off")


# 3. Bilateral Filter
plt.subplot(2, 4, 3)

plt.imshow(
    results["bilateral"],
    cmap="gray"
)

plt.title(
    "3. Bilateral Filter"
)

plt.axis("off")


# 4. Adaptive Canny
plt.subplot(2, 4, 4)

plt.imshow(
    results["canny"],
    cmap="gray"
)

plt.title(
    "4. Adaptive Canny"
)

plt.axis("off")


# 5. Morphological Closing
plt.subplot(2, 4, 5)

plt.imshow(
    results["morph"],
    cmap="gray"
)

plt.title(
    "5. Morphological Closing"
)

plt.axis("off")


# 6. Candidate Crack Regions
plt.subplot(2, 4, 6)

plt.imshow(
    cv2.cvtColor(
        results["candidate_img"],
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    "6. Detected Candidate Regions"
)

plt.axis("off")


# 7. Final Crack Detection
plt.subplot(2, 4, 7)

plt.imshow(
    cv2.cvtColor(
        results["result_img"],
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    "7. " + results["status"]
)

plt.axis("off")


plt.tight_layout()
plt.show()


# ============================================
# DISPLAY DETECTION VALUES
# ============================================

print("=" * 45)
print("WINDSHIELD CRACK DETECTION RESULT")
print("=" * 45)

print(
    "Adaptive Canny Lower Threshold :",
    results["lower_threshold"]
)

print(
    "Adaptive Canny Upper Threshold :",
    results["upper_threshold"]
)

print(
    "Edge Density                   :",
    round(
        results["edge_density"],
        4
    )
)

print(
    "Valid Crack Regions            :",
    results["valid_regions"]
)

print(
    "Long Contours                  :",
    results["long_contours"]
)

print(
    "Final Result                   :",
    results["status"]
)

FileNotFoundError: Image not found.

In [ ]:
# ============================================
# CELL 4 - TEST COMPLETE DATASET
# ============================================

cracked_folder = (
    "Dataset_Finalised/Cracked" #PUT PATH OF CRACKED WINDSHIELD IMAGES
)

non_cracked_folder = (
    "Dataset_Finalised/Non-Cracked" #PUT PATH OF NON-CRACKED/NORMAL WINDSHIELD IMAGES
)


actual = []
predicted = []


image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
)


# ============================================
# TEST CRACKED IMAGES
# Actual = 1
# ============================================

for filename in os.listdir(
    cracked_folder
):

    if filename.lower().endswith(
        image_extensions
    ):

        image_path = os.path.join(
            cracked_folder,
            filename
        )

        results = detect_crack(
            image_path
        )

        if results is not None:

            actual.append(1)

            predicted.append(
                results["prediction"]
            )


# ============================================
# TEST NON-CRACKED IMAGES
# Actual = 0
# ============================================

for filename in os.listdir(
    non_cracked_folder
):

    if filename.lower().endswith(
        image_extensions
    ):

        image_path = os.path.join(
            non_cracked_folder,
            filename
        )

        results = detect_crack(
            image_path
        )

        if results is not None:

            actual.append(0)

            predicted.append(
                results["prediction"]
            )


# ============================================
# DATASET SUMMARY
# ============================================

print(
    "Total images tested:",
    len(actual)
)

print(
    "Cracked images:",
    actual.count(1)
)

print(
    "Non-cracked images:",
    actual.count(0)
)

In [4]:
# ============================================
# CELL 5 - PERFORMANCE EVALUATION
# ============================================

cm = confusion_matrix(
    actual,
    predicted
)


# ============================================
# Extract Confusion Matrix Values
# ============================================

tn, fp, fn, tp = cm.ravel()


# ============================================
# Calculate Performance Metrics
# ============================================

accuracy = accuracy_score(
    actual,
    predicted
)

precision = precision_score(
    actual,
    predicted,
    zero_division=0
)

recall = recall_score(
    actual,
    predicted,
    zero_division=0
)

f1 = f1_score(
    actual,
    predicted,
    zero_division=0
)


# ============================================
# Display Numerical Results
# ============================================

print("=" * 50)

print(
    "WINDSHIELD CRACK DETECTION PERFORMANCE"
)

print("=" * 50)


print(
    "\nTotal Images Tested :",
    len(actual)
)

print(
    "\nTrue Positive (TP)  :",
    tp
)

print(
    "True Negative (TN)  :",
    tn
)

print(
    "False Positive (FP) :",
    fp
)

print(
    "False Negative (FN) :",
    fn
)


print(
    "\n--- PERFORMANCE METRICS ---"
)

print(
    f"Accuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-Score  : {f1 * 100:.2f}%"
)


# ============================================
# Display Confusion Matrix
# ============================================

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "No Crack",
        "Crack"
    ]
)

disp.plot(
    values_format="d"
)

plt.title(
    "Confusion Matrix - Windshield Crack Detection"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)

plt.show()

NameError: name 'actual' is not defined